# 12 - Modelos explicativos de precios publicados

- Pregunta: ¿cómo se asocia la distancia al transporte con el precio por m² entre publicaciones comparables?
- Se estiman modelos separados para venta y alquiler; los precios de ambas operaciones no se mezclan.
- Se compara la especificación de H3 con otra que controla además la fuente y una tercera que agrega el perfil de servicios/transporte de 11.
- Errores estándar agrupados por barrio reconocen dependencia territorial. El perfil de 11 se forma sin usar precios.
- Es un análisis explicativo observacional, no un pronóstico ni una estimación causal.
- Precios en USD son nominales; el corte de 2026 no permite estimar inflación o tendencias.

## 1. Muestra y variables

- Se conserva el universo y los filtros de calidad del notebook 08.
- Se excluye Airbnb y se requieren coordenadas en CABA, precio por m² válido, superficie y ambientes.
- Se modela log(precio por m²); superficie entra en logaritmo y ambientes como categoría.
- La segunda especificación agrega efectos fijos por fuente. La tercera agrega el perfil urbano de 11 y omite efectos fijos por barrio, porque el perfil es constante dentro de cada barrio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src import config, eda

REPORTS = config.REPORTS_DIR
base = pd.read_csv(
    config.PROCESSED_DIR / "df_publicaciones_enriquecidas_v1.csv",
    low_memory=False,
    dtype={"id_publicacion": str}
)
h3 = base[
    (base["fuente_norm"] != "airbnb")
    & base["coords_en_caba"]
    & base["precio_m2_usd"].notna()
    & ~base["precio_m2_usd_fuera_rango_flag"]
    & base["superficie_ref_m2"].notna()
    & base["ambientes_ref"].notna()
    & base["tipo_operacion_norm"].isin(["venta", "alquiler"])
].copy()
h3["lpm2"] = np.log(h3["precio_m2_usd"])
h3["lsup"] = np.log(h3["superficie_ref_m2"])
h3["amb_cat"] = h3["ambientes_ref"].clip(upper=5).astype(int).astype(str)
h3["dm"] = h3["dist_transporte_masivo_m"] / 100
clusters = pd.read_csv(config.PROCESSED_DIR / "pe3_clusters_barrios_v1.csv")
h3 = h3.merge(clusters[["barrio_oficial", "cluster"]], on="barrio_oficial", how="inner", validate="many_to_one")
h3 = h3.dropna(subset=["lpm2", "lsup", "dm", "barrio_oficial", "fuente_norm"])
display(h3.groupby(["tipo_operacion_norm", "fuente_norm"]).size().rename("avisos").to_frame())
print("Publicaciones utilizables:", len(h3))

## 2. Comparación de especificaciones

- **Referencia:** distancia, superficie, ambientes y barrio; replica el núcleo del análisis H3.
- **Extendida:** agrega fuente para reducir la posible confusión por diferencias de cobertura y producto entre portales.
- Se comparan tamaño de muestra, R² ajustado y criterio AIC.
- El coeficiente de distancia se convierte a variación porcentual por 100 m, con IC95%.
- La significancia no reemplaza la magnitud, la incertidumbre ni la plausibilidad del mecanismo.

In [ ]:
filas = []
modelos = {}
for op in ["venta", "alquiler"]:
    s = h3[h3["tipo_operacion_norm"] == op].copy()
    formula_base = "lpm2 ~ dm + lsup + C(amb_cat) + C(barrio_oficial)"
    formula_fuente = formula_base + " + C(fuente_norm)"
    # Especificación entre perfiles: no incluye efectos fijos de barrio, que absorberían el cluster.
    formula_perfil = "lpm2 ~ dm + lsup + C(amb_cat) + C(fuente_norm) + C(cluster)"
    m_base = eda.ols_cluster(s, formula_base, "barrio_oficial")
    m_fuente = eda.ols_cluster(s, formula_fuente, "barrio_oficial")
    m_perfil = eda.ols_cluster(s, formula_perfil, "barrio_oficial")
    modelos[op] = m_perfil

    for nombre, modelo in [("H3 base", m_base), ("H3 + fuente", m_fuente), ("Perfil urbano + fuente", m_perfil)]:
        beta = modelo.params["dm"]
        lo, hi = modelo.conf_int().loc["dm"]
        filas.append({
            "operacion": op, "especificacion": nombre, "n": int(modelo.nobs),
            "barrios": s["barrio_oficial"].nunique(),
            "efecto_pct_por_100m": 100 * (np.exp(beta) - 1),
            "ic95_inf_pct": 100 * (np.exp(lo) - 1),
            "ic95_sup_pct": 100 * (np.exp(hi) - 1),
            "p": modelo.pvalues["dm"], "r2_ajustado": modelo.rsquared_adj,
            "aic": modelo.aic
        })

tabla_modelos = pd.DataFrame(filas)
display(tabla_modelos.round(4))
tabla_modelos.to_csv(REPORTS / "pe3_modelos_explicativos_precios_v1.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 3.8))
orden = np.arange(len(tabla_modelos))
ef = tabla_modelos["efecto_pct_por_100m"].to_numpy()
err_inf = ef - tabla_modelos["ic95_inf_pct"].to_numpy()
err_sup = tabla_modelos["ic95_sup_pct"].to_numpy() - ef
colores = ["#4b78a8" if x == "venta" else "#6b8f71" for x in tabla_modelos["operacion"]]
for i in orden:
    ax.errorbar(ef[i], i, xerr=[[err_inf[i]], [err_sup[i]]], fmt="o",
                color=colores[i], capsize=3)
ax.axvline(0, color="#777777", linewidth=1)
ax.set_yticks(orden)
ax.set_yticklabels(tabla_modelos["operacion"] + " - " + tabla_modelos["especificacion"])
ax.set_xlabel("Variación asociada por 100 m más lejos (%)")
ax.set_title("Asociación ajustada entre distancia y precio publicado (IC95%)")
plt.tight_layout()
plt.show()

## 3. Interpretación y límites

- Si el signo o magnitud cambia al agregar efectos de fuente, la composición de portales importa para H3.
- Los efectos fijos por barrio absorben diferencias promedio entre barrios; no eliminan toda confusión dentro de cada barrio.
- El resultado se refiere a precios pedidos observados y a avisos geocodificados, no a transacciones.
- Distancias en línea recta son proxies de accesibilidad y no miden tiempo de viaje, frecuencia ni calidad del servicio.
- Los errores agrupados y los intervalos cuantifican incertidumbre estadística, pero no resuelven sesgos de selección ni la falta de snapshots.
- No corresponde traducir el coeficiente en causalidad o efecto de una obra de transporte.